# 从三点拟合到 PCA
先修：矩阵乘法、内积和正文的正交投影。按顺序运行，可直接修改数据；本实验不锁定输入。

我们依次研究拟合、分解、压缩与扰动。先预测：一条直线无法穿过三点时，残差能否与设计矩阵的每列垂直？

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6, suppress=True)
X = np.array([[1., 0.], [1., 1.], [1., 2.]])
y = np.array([1., 2., 2.])
beta = np.linalg.lstsq(X, y, rcond=None)[0]
fitted = X @ beta
residual = y - fitted
print('系数、拟合、残差：', beta, fitted, residual, sep='\n')
print('残差内积：', X.T @ residual)

默认三点数据的系数是斜率 $1/2$、截距 $7/6$。修改数据后，解析数值也随之改变；几何条件 $X^Tr=0$ 仍是最小二乘的共同性质。下面以观测序号画出拟合与残差，适用于重复列情形。

In [ ]:
fig, ax = plt.subplots()
index = np.arange(len(y))
ax.scatter(index, y, label='observed')
ax.plot(index, fitted, label='fitted')
ax.vlines(index, y, fitted, color='tab:red', label='residual')
ax.set(xlabel='observation index', ylabel='response')
ax.legend(); plt.show()
print('残差平方和：', residual @ residual)

## 满列秩与秩亏：公式的条件在哪里
满列秩时，QR 的三角回代和正规方程都能得到唯一系数。秩亏时，某个奇异方向不再提供信息；伪逆将该方向的系数设为零，得到最小范数解。

数值计算以相对于最大奇异值的阈值辨认零方向。这里统一取 $r=\epsilon\max(m,n)$，保留 $\sigma_j>r\sigma_1$ 的方向；这是一种数值秩约定，并不证明小奇异值在精确算术下等于零。修改原始 $X$ 为重复列后，从头运行仍能观察两种情形。


In [ ]:
U, singular, Vt = np.linalg.svd(X, full_matrices=False)
relative_cutoff = np.finfo(float).eps * max(X.shape)
threshold = relative_cutoff * singular[0]
keep = singular > threshold
inverse_singular = np.zeros_like(singular)
inverse_singular[keep] = 1 / singular[keep]
beta_svd = Vt.T @ (inverse_singular * (U.T @ y))
beta_pinv = np.linalg.pinv(X, rcond=relative_cutoff) @ y
beta_lstsq = np.linalg.lstsq(X, y, rcond=relative_cutoff)[0]
print('奇异值、阈值、保留方向：', singular, threshold, keep)
print('SVD / 伪逆 / 最小二乘系数：', beta_svd, beta_pinv, beta_lstsq, sep='\n')
if keep.sum() == X.shape[1]:
    Q, R = np.linalg.qr(X, mode='reduced')
    beta_qr = np.linalg.solve(R, Q.T @ y)
    print('Q, R：', Q, R, sep='\n')
    print('QR 系数：', beta_qr)
    # 正规方程平方条件数，近共线时可能在浮点计算中已不可逆。
    try:
        beta_normal = np.linalg.solve(X.T @ X, X.T @ y)
        print('正规方程系数：', beta_normal)
    except np.linalg.LinAlgError:
        print('正规方程在当前精度下不可逆；上方 SVD 仍直接作用于 X。')
else:
    print('秩亏：普通三角求逆和正规方程求逆不适用；使用保留方向的伪逆。')
rank_one = singular[0] * np.outer(U[:, 0], Vt[0])
print('秩一误差与第二奇异值：', np.linalg.norm(X-rank_one, 2), singular[1])

## PCA 与训练样本白化
下面生成总体协方差为 $\left(\begin{smallmatrix}2&1\\1&2\end{smallmatrix}\right)$ 的二维样本。总体首方向是 $(1,1)/\sqrt2$，解释比例为 75%；样本结果会有随机误差。改变样本数，看它怎样变化。

In [ ]:
rng = np.random.default_rng(2026)
population = np.array([[2., 1.], [1., 2.]])
train = rng.multivariate_normal([0, 0], population, size=80)
center = train.mean(axis=0)
Z = train - center
S = Z.T @ Z / (len(Z)-1)
eigenvalues, directions = np.linalg.eigh(S)
order = np.argsort(eigenvalues)[::-1]
eigenvalues, directions = eigenvalues[order], directions[:, order]
scores = Z @ directions
reconstructed = np.outer(scores[:, 0], directions[:, 0])
white = scores / np.sqrt(eigenvalues)
new = rng.multivariate_normal([0, 0], population, size=2000)
white_new = ((new-center) @ directions) / np.sqrt(eigenvalues)
print('解释比例：', eigenvalues / eigenvalues.sum())
print('舍弃方向的样本平方误差：', np.sum((Z-reconstructed)**2)/(len(Z)-1))
print('训练样本白化协方差：', np.cov(white, rowvar=False))
print('新样本白化协方差：', np.cov(white_new, rowvar=False))
fig, ax = plt.subplots()
ax.scatter(Z[:, 0], Z[:, 1], alpha=.5, label='centered')
ax.scatter(reconstructed[:, 0], reconstructed[:, 1], s=12, label='rank one')
ax.axis('equal'); ax.legend(); plt.show()

训练协方差为单位阵是代数恒等式；新样本没有这一保证。这里训练与新样本来自同一总体，差别仍会由训练估计误差产生。新样本数增大只能减少新样本自身的抽样误差，不能消除已固定的训练变换误差。

In [ ]:
D = np.diag([1., 1e-6])
b = np.array([1., 0.])
db = np.array([0., 1e-6])
exact = np.linalg.solve(D, b)
changed = np.linalg.solve(D, b+db)
print('输入相对变化：', np.linalg.norm(db)/np.linalg.norm(b))
print('解相对变化：', np.linalg.norm(changed-exact)/np.linalg.norm(exact))
print('条件数：', np.linalg.cond(D))

## 统一缩放与相对条件数
比较 $10^{-12}I$ 和 $\operatorname{diag}(10^{12},1)$。前者只是整体数值小；后者对不同方向的放大倍数悬殊。高相对条件数不等于每次扰动都达到最坏放大。

In [ ]:
for matrix in [1e-12*np.eye(2), np.diag([1e12, 1.])]:
    print('奇异值：', np.linalg.svd(matrix, compute_uv=False), '条件数：', np.linalg.cond(matrix))

## 自己试一试
1. 把设计矩阵的第二列改成第一列。系数和拟合值是否仍唯一？
2. 只保留较小奇异值，秩一近似误差怎样变化？
3. 将上格扰动换到第一坐标，是否仍放大一百万倍？

先作答，再运行下格参考。

In [ ]:
duplicate = np.column_stack([np.ones(len(y)), np.ones(len(y))])
minimum = np.linalg.lstsq(duplicate, y, rcond=None)[0]
alternative = minimum + np.array([1., -1.])
small_only = singular[1] * np.outer(U[:, 1], Vt[1])
print('两个系数：', minimum, alternative)
print('拟合差：', duplicate @ (minimum-alternative))
print('只保留较小项的误差：', np.linalg.norm(X-small_only, 2))
db_first = np.array([1e-6, 0.])
print('第一方向放大：', np.linalg.norm(np.linalg.solve(D, db_first))/np.linalg.norm(db_first))

系数可沿零空间移动，拟合仍唯一。保留较小项时误差等于最大奇异值；第一方向放大为 1。高条件数描述最坏方向，不能解释为每次误差都同样放大。